for dbt automation later

In [0]:
%pip install dbt-core==1.11.6 dbt-databricks==1.11.5


  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 30.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 779.2/779.2 kB 43.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 80.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 70.6 MB/s eta 0:00:00
  Created wheel for thrift: filename=thrift-0.20.0-cp312-cp312-linux_aarch64.whl size=409740 sha256=6bf9895cec7196d92970fdc52b9780af6d7a9ddfaf6a982fe59a90130f865bc5
  Stored in directory: /home/spark-7be63e82-30d2-401f-b693-e7/.cache/pip/wheels/57/4d/9e/539ef8e052f8e2db244ddf8ac1f2e623b4663f2ee233ffafb1
Successfully built thrift
  Attempting uninstall: protobuf
    Found existing installation: protobuf 5.29.4
    Not uninstalling protobuf at /databricks/python3/lib/python3.12/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-7be63e82-30d2-401f-b693-e71

catalog and schema

In [0]:
%sql
USE CATALOG rearc;
USE SCHEMA rearc_schema;

volume pointing to external location

In [0]:
%sql
CREATE EXTERNAL VOLUME IF NOT EXISTS rearc.rearc_schema.raw_vol
LOCATION 's3://rearc-data-quest-soham-48291/raw/';


data loading for data usa

In [0]:
%sql
DROP TABLE IF EXISTS rearc.rearc_schema.stg_population_raw;

CREATE TABLE rearc.rearc_schema.stg_population_raw AS
SELECT
  CAST(d.`Year` AS INT)         AS year,
  CAST(d.`Population` AS BIGINT) AS population,
  d.`Nation ID`                 AS nation_id,
  d.`Nation`                    AS nation
FROM (
  SELECT explode(data) AS d
  FROM read_files(
    '/Volumes/rearc/rearc_schema/raw_vol/datausa/population_latest.json',
    format => 'json'
  )
);


num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT count(*) FROM rearc.rearc_schema.stg_population_raw;


count(*)
10


In [0]:
%sql
SELECT * FROM rearc.rearc_schema.stg_population_raw LIMIT 5;

year,population,nation_id,nation
2013,316128839,01000US,United States
2014,318857056,01000US,United States
2015,321418821,01000US,United States
2016,323127515,01000US,United States
2017,325719178,01000US,United States


data laodingg for bls

In [0]:
%sql
DROP TABLE IF EXISTS rearc.rearc_schema.stg_bls_pr_data_current;

CREATE TABLE rearc.rearc_schema.stg_bls_pr_data_current AS
SELECT
  trim(_c0)                      AS series_id,
  try_cast(trim(_c1) AS INT)     AS year,
  trim(_c2)                      AS period,
  try_cast(trim(_c3) AS DOUBLE)  AS value,
  trim(_c4)                      AS footnote_codes
FROM read_files(
  '/Volumes/rearc/rearc_schema/raw_vol/bls/pr/pr.data.0.Current',
  format => 'csv',
  sep => '\t',
  header => 'false'
)
WHERE lower(trim(_c0)) <> 'series_id';


num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT count(*) FROM rearc.rearc_schema.stg_bls_pr_data_current;

count(*)
37521


In [0]:
%sql
SELECT * FROM rearc.rearc_schema.stg_bls_pr_data_current LIMIT 5;

series_id,year,period,value,footnote_codes
PRS30006011,1995,Q01,2.6,null
PRS30006011,1995,Q02,2.1,null
PRS30006011,1995,Q03,0.9,null
PRS30006011,1995,Q04,0.1,null
PRS30006011,1995,Q05,1.4,null


Data Platform Notes

Metadata-driven pipeline: minimal config-table pattern implemented for staging ingestion.

Auto Loader: not used in final run due free-edition scope; read_files fallback used.

Governance: Unity Catalog + external location + volume + table metadata.

Schema evolution: tolerant parsing (try_cast) + null-quality checks.

Partition evolution: v1/v2 table cutover strategy via stable view.

Governance

In [0]:
%sql
ALTER TABLE rearc.rearc_schema.stg_population_raw SET TBLPROPERTIES (
  layer = 'staging',
  domain = 'population',
  data_owner = 'soham'
);

ALTER TABLE rearc.rearc_schema.stg_bls_pr_data_current SET TBLPROPERTIES (
  layer = 'staging',
  domain = 'bls',
  data_owner = 'soham'
);

COMMENT ON TABLE rearc.rearc_schema.stg_population_raw IS 'Staging table from datausa population';
COMMENT ON TABLE rearc.rearc_schema.stg_bls_pr_data_current IS 'Stagin table for bls pr.data.0.Current';

data quality validation 

In [0]:
%sql
select
  sum(case when year is null then 1 else 0 end) as null_year_rows,
  sum(case when population is null then 1 else 0 end) as null_population_rows
from rearc.rearc_schema.stg_population_raw

null_year_rows,null_population_rows
0,0


In [0]:
%sql
select
sum(case when year is null then 1 else 0 end) as null_year_rows,
sum(case when value is null then 1 else 0 end) as null_value_rows,
sum(case when period is null or trim(period) = '' then 1 else 0 end) as null_or_blank_period_rows
from rearc.rearc_schema.stg_bls_pr_data_current

null_year_rows,null_value_rows,null_or_blank_period_rows
0,0,0


metadata-driven ingestion

in above we have statically defined the tables

rearc.rearc_schema.stg_bls_pr_data_current
rearc.rearc_schema.stg_population_raw

but the below showcases how we will handle it in production making table creation entirely dynamic in nature

In [0]:
%sql
DROP TABLE IF EXISTS rearc.rearc_schema.ingestion_config;
CREATE TABLE IF NOT EXISTS rearc.rearc_schema.ingestion_config (
  source_name STRING,
  source_path STRING,
  format STRING,
  target_table STRING,
  options STRING
);

DELETE FROM ingestion_config;

INSERT INTO ingestion_config VALUES
('population_api','/Volumes/rearc/rearc_schema/raw_vol/datausa/population_latest.json','json','rearc.rearc_schema.stg_population_raw_cfg','{"explode_data": true}'),
('bls_pr_data_current','/Volumes/rearc/rearc_schema/raw_vol/bls/pr/pr.data.0.Current','csv','rearc.rearc_schema.stg_bls_pr_data_current_cfg','{"sep":"\\t","header":"false","drop_header_row":true}');


num_affected_rows,num_inserted_rows
2,2


In [0]:
%python
import json

spark.sql("USE CATALOG rearc")
spark.sql("USE SCHEMA rearc_schema")

cfg = spark.table("rearc.rearc_schema.ingestion_config").collect()

for r in cfg:
    source_path = r["source_path"]
    fmt = r["format"]
    target = r["target_table"]
    opts = json.loads(r["options"] or "{}")

    spark.sql(f"DROP TABLE IF EXISTS {target}")

    if fmt == "json":
        spark.sql(f"""
        CREATE TABLE {target} AS
        SELECT
          CAST(d.`Year` AS INT)          AS year,
          CAST(d.`Population` AS BIGINT) AS population,
          d.`Nation ID`                  AS nation_id,
          d.`Nation`                     AS nation
        FROM (
          SELECT explode(data) AS d
          FROM read_files('{source_path}', format => 'json')
        )
        """)
    elif fmt == "csv":
        spark.sql(f"""
        CREATE TABLE {target} AS
        SELECT
          trim(_c0)                      AS series_id,
          try_cast(trim(_c1) AS INT)     AS year,
          trim(_c2)                      AS period,
          try_cast(trim(_c3) AS DOUBLE)  AS value,
          trim(_c4)                      AS footnote_codes
        FROM read_files(
          '{source_path}',
          format => 'csv',
          sep => '\\t',
          header => 'false'
        )
        WHERE lower(trim(_c0)) <> 'series_id'
        """)
    else:
        raise ValueError(f"Unsupported format: {fmt}")

print("Metadata-driven ingestion complete")


Metadata-driven ingestion complete


In [0]:
%sql
SELECT count(*) FROM rearc.rearc_schema.stg_population_raw_cfg;


count(*)
10


In [0]:
%sql
SELECT count(*) FROM rearc.rearc_schema.stg_bls_pr_data_current_cfg;

count(*)
37521


Auto loader Note

Auto Loader was evaluated but not used in the final Free Edition notebook path.

For this task read_files(...) was used as the ingestion fallback 

In production environment, this would be replaced by Auto Loader with checkpoint + schema location.

Partition Evolution

The current dataset is relatively small, so the report tables are kept simple to avoid unnecessary partition overhead. 

For production, the preferred evolution path is to shift from rigid static partitioning to Liquid Clustering  so table layout can adapt to changing query patterns. 

If physical layout needs to change, a versioned cutover approach will be used (v1 -> v2) with downstream consumers abstracted via a stable view to avoid breaking report queries.


dbt automation

##### we can use databricks secrets for production
##### but here we rotate the pat 

In [0]:
%sh
set -e
mkdir -p ~/.dbt
cat > ~/.dbt/profiles.yml <<'YAML'
rearc_dbt:
  target: dev
  outputs:
    dev:
      type: databricks
      method: http
      host: https://dbc-b63d8b95-3ff1.cloud.databricks.com
      http_path: /sql/1.0/warehouses/0eac97d4236672d5
      token: dapi7050088d98a81bed51b56fjc3d4b71571
      catalog: rearc
      schema: rearc_dbt_schema
      threads: 1
YAML

In [0]:
%sh
cd /Workspace/Repos/sohamkundu511@gmail.com/data_quest/rearc_dbt
dbt debug --profiles-dir ~/.dbt --target dev

12:33:39  Running with dbt=1.11.6
12:33:39  dbt version: 1.11.6
12:33:39  python version: 3.12.3
12:33:39  python path: /local_disk0/.ephemeral_nfs/envs/pythonEnv-7be63e82-30d2-401f-b693-e71dae6b5916/bin/python
12:33:39  os info: Linux-5.15.0-1100-aws-aarch64-with-glibc2.39


12:33:41  Using profiles dir at /home/spark-7be63e82-30d2-401f-b693-e7/.dbt
12:33:41  Using profiles.yml file at /home/spark-7be63e82-30d2-401f-b693-e7/.dbt/profiles.yml
12:33:41  Using dbt_project.yml file at /Workspace/Repos/sohamkundu511@gmail.com/data_quest/rearc_dbt/dbt_project.yml
12:33:41  adapter type: databricks
12:33:41  adapter version: 1.11.5
12:33:41  Configuration:
12:33:41    profiles.yml file [OK found and valid]
12:33:41    dbt_project.yml file [OK found and valid]
12:33:41  Required dependencies:
12:33:42   - git [OK found]

12:33:42  Connection:
12:33:42    host: https://dbc-b63d8b95-3ff1.cloud.databricks.com
12:33:42    http_path: /sql/1.0/warehouses/0eac97d4236672d5
12:33:42    catalog: rearc
12:33:42    schema: rearc_dbt_schema
12:33:42  Registered adapter: databricks=1.11.5
12:33:42  [WARNING]: Use managed Iceberg tables when table_format is iceberg. When this flag is disabled, UniForm is used instead.
You may opt into the new behavior sooner by setting `flags.us

In [0]:
%sh
set -e
cd /Workspace/Repos/sohamkundu511@gmail.com/data_quest/rearc_dbt
dbt deps --profiles-dir ~/.dbt --target dev
dbt run --profiles-dir ~/.dbt --target dev
dbt test --profiles-dir ~/.dbt --target dev

12:35:47  Running with dbt=1.11.6
12:35:47  Warning: No packages were found in packages.yml
12:35:47  Warning: No packages were found in packages.yml
12:35:49  Running with dbt=1.11.6


12:35:51  Registered adapter: databricks=1.11.5
12:35:51  [WARNING]: Use managed Iceberg tables when table_format is iceberg. When this flag is disabled, UniForm is used instead.
You may opt into the new behavior sooner by setting `flags.use_managed_iceberg` to `True` in `dbt_project.yml`.
Visit https://docs.getdbt.com/reference/global-configs/behavior-changes for more information.
12:35:51  [WARNING]: Configuration paths exist in your dbt_project.yml file which do not apply to any resources.
There are 1 unused configuration paths:
- models.rearc_dbt.example
12:35:52  Found 3 models, 5 data tests, 2 sources, 731 macros
12:35:52  
12:35:52  Concurrency: 1 threads (target='dev')
12:35:52  
12:35:53  1 of 3 START sql view model rearc_dbt_schema.report_3 .......................... [RUN]
12:35:53  [WARNING]: Use revamped materializations based on separating create and insert.  This allows more performant column comments, as well as new column features.
You may opt into the new behavior soon

12:36:00  Registered adapter: databricks=1.11.5
12:36:00  [WARNING]: Use managed Iceberg tables when table_format is iceberg. When this flag is disabled, UniForm is used instead.
You may opt into the new behavior sooner by setting `flags.use_managed_iceberg` to `True` in `dbt_project.yml`.
Visit https://docs.getdbt.com/reference/global-configs/behavior-changes for more information.
12:36:01  [WARNING]: Configuration paths exist in your dbt_project.yml file which do not apply to any resources.
There are 1 unused configuration paths:
- models.rearc_dbt.example
12:36:02  Found 3 models, 5 data tests, 2 sources, 731 macros
12:36:02  
12:36:02  Concurrency: 1 threads (target='dev')
12:36:02  
12:36:02  1 of 5 START test source_not_null_rearc_schema_stg_bls_pr_data_current_period .. [RUN]
12:36:03  1 of 5 PASS source_not_null_rearc_schema_stg_bls_pr_data_current_period ........ [PASS in 0.70s]
12:36:03  2 of 5 START test source_not_null_rearc_schema_stg_bls_pr_data_current_series_id  [RUN]
1